<a href="https://colab.research.google.com/github/luquelab/Flyby-Denial/blob/draft/FINAL_FINAL_FINAL_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and import packages

In [ ]:
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import collections # Added for Counter
import zipfile
import os

from google.colab import files
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler

# Install Biopython
!pip install biopython

# Install xlsxwriter for Excel export
!pip install xlsxwriter

from Bio import SeqIO # Import SeqIO for parsing FASTA files
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio import ExPASy
from Bio import SwissProt
from Bio import Align
from Bio.Seq import Seq # Added for creating Seq objects
from Bio.SeqRecord import SeqRecord # Added for creating SeqRecord objects

from IPython.display import display, HTML
import ipywidgets as widgets
from Bio import Entrez
from google.colab import userdata # For API key if needed

# Input data

## Option 1: Upload a FASTA file
You can upload one or more FASTA files from your local machine using the following code:

In [ ]:

# Upload one or more FASTA files
print("Please select your FASTA files for upload:")
uploaded = files.upload()

# This list will store all SeqRecord objects from all uploaded FASTA files
all_fasta_sequences = []

print("\n--- Processing uploaded FASTA files ---")
for filename in uploaded.keys():
    print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')
    with open(filename, 'wb') as f:
        f.write(uploaded[filename])

    try:
        # Read sequences from the current FASTA file
        sequences_from_file = list(SeqIO.parse(filename, "fasta"))
        print(f"Loaded {len(sequences_from_file)} sequences from {filename}")
        all_fasta_sequences.extend(sequences_from_file) # Add to the master list

        # Print a few details for confirmation
        for i, seq_record in enumerate(sequences_from_file[:3]): # Show first 3 sequences from this file
            print(f"  Sample from {filename}: ID: {seq_record.id}, Length: {len(seq_record.seq)}")
            if len(seq_record.seq) > 50:
                print(f"  Sequence preview: {str(seq_record.seq)[:50]}...\n")
            else:
                print(f"  Sequence preview: {str(seq_record.seq)}\n")

    except Exception as e:
        print(f"Error reading FASTA file {filename}: {e}")

print(f"Total sequences loaded from all uploaded FASTA files: {len(all_fasta_sequences)} SeqRecords.")

# Now, `all_fasta_sequences` contains SeqRecord objects from all uploaded files and can be used by subsequent cells.


## Option 2: Enter protein sequence manually

If you have a protein sequence readily available, you can directly paste it here. Make sure it's a raw protein sequence (amino acid letters only, no FASTA headers).

In [ ]:


# Ensure all_fasta_sequences exists and is a list
if 'all_fasta_sequences' not in globals() or not isinstance(all_fasta_sequences, list):
    all_fasta_sequences = []
    print("Initialized 'all_fasta_sequences' list for manually entered sequences.")

print("Please enter your protein sequences one by one. Press Enter on an empty line to finish.")

sequence_count = 0
while True:
    user_sequence_input = input(f"Enter protein sequence (or press Enter to finish): ")

    if not user_sequence_input.strip(): # Check for empty input to break the loop
        print("No more sequences to add. Finishing manual input.")
        break

    # Clean the input sequence (remove spaces, newlines, etc.)
    cleaned_sequence = user_sequence_input.strip().replace(' ', '').replace('\n', '').replace('\r', '')

    if cleaned_sequence:
        try:
            sequence_count += 1
            # Create a Seq object
            protein_seq = Seq(cleaned_sequence)
            # Create a SeqRecord object. Assign a generic ID if no specific one is provided.
            new_seq_record = SeqRecord(protein_seq, id=f"User_Input_Protein_{sequence_count}", description="Manually entered sequence")

            all_fasta_sequences.append(new_seq_record)
            print(f"  Added sequence {sequence_count}: ID: {new_seq_record.id}, Length: {len(new_seq_record.seq)}")
            if len(new_seq_record.seq) > 50:
                print(f"  Sequence preview: {str(new_seq_record.seq)[:50]}...\n")
            else:
                print(f"  Sequence preview: {str(new_seq_record.seq)}\n")

        except Exception as e:
            print(f"Error processing manual sequence input: {e}")
    else:
        print("No valid sequence entered. Please try again or press Enter to finish.")

print(f"Total sequences loaded from manual input: {sequence_count} SeqRecords. Current total in all_fasta_sequences: {len(all_fasta_sequences)}.")

# Note: Previous content for fetching from NCBI has been removed as per user request.

## Option 3: Fetch from NCBI by Organism and Select FASTA files

This option allows you to search for protein sequences from the NCBI database by specifying an organism name. You can then view the results and select which sequences you'd like to include in your analysis.

In [ ]:


# --- Entrez Configuration ---
# IMPORTANT: NCBI's E-utilities require an email address for ALL requests.
# Please replace "your.email@example.com" with your actual email address below.
# This is a one-time setup for NCBI compliance and is not interactive.
Entrez.email = "kennedykreutzer15@gmail.com" # <--- REPLACE THIS WITH YOUR EMAIL

# Optionally, set an API key for higher request limits. (Highly Recommended)
# Get an API key from: https://ncbi.nlm.nih.gov/account/settings/
# You can store it in Colab's secrets and retrieve it like: userdata.get('NCBI_API_KEY')
# Entrez.api_key = userdata.get('NCBI_API_KEY') # Uncomment if you have an API key from Colab secrets

# Initialize the global list for all sequences (will be populated upon fetching)
if 'all_fasta_sequences' not in globals() or not isinstance(all_fasta_sequences, list):
    all_fasta_sequences = []
    print("Initialized 'all_fasta_sequences' list for fetched sequences.")

# Global list to store IDs of proteins that the user has finally selected
selected_ids_to_fetch = [] # Renamed from protein_ids_to_fetch to match user's request and downstream usage

# Global temporary list to store all fetched sequences before user selection
# _all_fetched_sequences_temp = [] # No longer needed with new fetching logic

# --- Widgets for Search Parameters ---
organism_widget = widgets.Text(description='Organism Name:', placeholder='e.g., Homo sapiens, Mus musculus', layout=widgets.Layout(width='500px'))
num_proteins_widget = widgets.IntText(description='Max Proteins:', value=100, min=1, max=1000, layout=widgets.Layout(width='200px'))
fetch_search_button = widgets.Button(description='Search & Display Proteins', button_style='info')
search_output = widgets.Output()

# --- Widgets for Filter and Selection (initially hidden) ---
function_filter_input_widget = widgets.Text(description='Filter Description:', placeholder='e.g., kinase, receptor', layout=widgets.Layout(width='500px'))
apply_filter_button = widgets.Button(description='Apply Filter', button_style='primary')
proteins_selector_widget = widgets.SelectMultiple(description='Select Proteins:', options=[], rows=5, layout=widgets.Layout(width='500px'))
confirm_selection_button = widgets.Button(description='Confirm Selection', button_style='success')
filter_selection_output = widgets.Output()

# Group filter/selection widgets and initially hide them
filter_selection_widgets_box = widgets.VBox(
    [function_filter_input_widget, apply_filter_button, proteins_selector_widget, confirm_selection_button, filter_selection_output],
    layout={'display': 'none'} # Hidden until search results are available
)

# --- Main Display Layout ---
display(organism_widget, num_proteins_widget, fetch_search_button, search_output, filter_selection_widgets_box)

current_search_results = [] # To hold ID and description for filtering and selection

def update_proteins_selector(filter_text=""):
    """Updates the proteins_selector_widget options based on a filter text."""
    options_for_selector = []
    for item in current_search_results:
        if not filter_text or filter_text.lower() in item['description'].lower() or filter_text.lower() in item['id'].lower():
            options_for_selector.append((f"{item['id']} - {item['description']}", item['id']))
    proteins_selector_widget.options = options_for_selector

def on_fetch_search_button_click(b):
    global current_search_results
    global selected_ids_to_fetch

    with search_output:
        search_output.clear_output()
        filter_selection_output.clear_output()
        current_search_results = [] # Clear previous search results
        selected_ids_to_fetch = [] # Clear previous selection
        # _all_fetched_sequences_temp = [] # Not needed with new fetching logic

        organism_name = organism_widget.value.strip()
        num_proteins = num_proteins_widget.value

        if not organism_name:
            print("Please enter an Organism Name to search.")
            return

        print(f"Searching for protein IDs for '{organism_name}' (max {num_proteins})...")
        try:
            # Search for IDs related to the organism
            handle = Entrez.esearch(db="protein", term=f"{organism_name}[Organism]", retmax=str(num_proteins))
            record = Entrez.read(handle)
            handle.close()
            id_list = record["IdList"]

            if not id_list:
                print(f"No protein sequences found for '{organism_name}'. Please try a different organism or check spelling.")
                filter_selection_widgets_box.layout.display = 'none' # Hide widgets if no results
                return

            print(f"Found {len(id_list)} protein IDs. Fetching summaries...")
            handle_summary = Entrez.esummary(db="protein", id=",".join(id_list))
            summary_records = Entrez.read(handle_summary)
            handle_summary.close()

            for s_rec in summary_records:
                protein_id = str(s_rec['Id'])
                description = s_rec.get('Title', 'No title available')
                current_search_results.append({'id': protein_id, 'description': description})

            # Update proteins_selector_widget options with all fetched results
            update_proteins_selector() # Call without filter to show all initially
            filter_selection_widgets_box.layout.display = 'block' # Show filter/selection widgets

            print(f"Found {len(current_search_results)} proteins. Select them and click 'Confirm Selection' to fetch FASTA sequences.")

        except Exception as e:
            print(f"An error occurred during NCBI search or summary fetch: {e}")
            filter_selection_widgets_box.layout.display = 'none' # Hide widgets on error

def on_apply_filter_button_click(b):
    with filter_selection_output:
        filter_selection_output.clear_output()
        filter_text = function_filter_input_widget.value.strip()
        update_proteins_selector(filter_text)
        print(f"Applied filter: '{filter_text}'. {len(proteins_selector_widget.options)} proteins displayed.")

def on_confirm_selection_button_click(b):
    global selected_ids_to_fetch
    global all_fasta_sequences # Now populate the main list
    with filter_selection_output:
        filter_selection_output.clear_output()
        selected_ids_to_fetch = list(proteins_selector_widget.value) # Get selected IDs from the widget

        # Removed: all_fasta_sequences = [] # This line was causing the list to be cleared

        if selected_ids_to_fetch:
            print(f"Confirmed {len(selected_ids_to_fetch)} protein(s) for analysis: {', '.join(selected_ids_to_fetch)}")
            print(f"Fetching FASTA sequences for {len(selected_ids_to_fetch)} selected proteins...")
            try:
                # Perform the actual efetch for only the selected IDs
                fasta_handle = Entrez.efetch(db="protein", id=",".join(selected_ids_to_fetch), rettype="fasta", retmode="text")
                fasta_data = fasta_handle.read()
                fasta_handle.close()

                from io import StringIO
                fetched_sequences = list(SeqIO.parse(StringIO(fasta_data), "fasta"))

                all_fasta_sequences.extend(fetched_sequences)
                print(f"Successfully fetched and stored {len(fetched_sequences)} sequences. Total sequences in all_fasta_sequences: {len(all_fasta_sequences)}.")
                for i, seq_record in enumerate(fetched_sequences[:3]): # Show first 3 sequences
                    print(f"  Selected Sample: ID: {seq_record.id}, Length: {len(seq_record.seq)}")
                    if len(seq_record.seq) > 50:
                        print(f"  Sequence preview: {str(seq_record.seq)[:50]}...\n")
                    else:
                        print(f"  Sequence preview: {str(seq_record.seq)}\n")
            except Exception as e:
                print(f"An error occurred during NCBI FASTA fetching for selected proteins: {e}")
                # If an error occurs, the list should not be cleared by this specific action
        else:
            print("No proteins selected for analysis. 'all_fasta_sequences' remains unchanged by this action.")
            # Removed: all_fasta_sequences = [] # This line was causing the list to be cleared if nothing was selected

# Attach button click handlers
fetch_search_button.on_click(on_fetch_search_button_click)
apply_filter_button.on_click(on_apply_filter_button_click)
confirm_selection_button.on_click(on_confirm_selection_button_click)

print("--- Enter organism name in the text box above, then click 'Search & Display Proteins' to begin. ---")

# Calculate Physicochemical Properties of Proteins

Using Biopython's `ProtParam` module, we can calculate various physicochemical properties of a protein sequence, such as its amino acid length, molecular weight, amino acid composition, isoelectric point, net charge, and extinction coefficient.

In [ ]:
sequences_for_analysis_prep = []
# This list will store dictionaries, each containing protein info, its ProteinAnalysis object, and its properties
protein_data_for_analysis = []

# --- Rely solely on all_fasta_sequences which should now contain both uploaded and fetched sequences ---
if 'all_fasta_sequences' in globals() and all_fasta_sequences: # Check in globals() as it's set by a previous cell
    print("Preparing protein sequences from 'all_fasta_sequences' for analysis.")
    for seq_record in all_fasta_sequences:
        sequences_for_analysis_prep.append({'id': seq_record.id, 'sequence': str(seq_record.seq)})
else:
    print("No protein sequences available for analysis. Please run the protein fetching or upload cells first.")


if sequences_for_analysis_prep:
    print(f"Found {len(sequences_for_analysis_prep)} protein(s) for physicochemical property analysis.")
    # Prepare the protein_data_for_analysis list with ProteinAnalysis objects
    for seq_info in sequences_for_analysis_prep:
        identifier = seq_info['id']
        protein_sequence_string = seq_info['sequence']

        if protein_sequence_string:
            analysed_seq_obj = ProteinAnalysis(protein_sequence_string)
            protein_data_for_analysis.append({
                'id': identifier,
                'sequence': protein_sequence_string,
                'analysis_obj': analysed_seq_obj,
                'properties': {} # Dictionary to store calculated properties for this protein
            })
        else:
            print(f"Skipping preparation for {identifier} due to empty sequence.")

print("\nProtein preparation complete. Proceeding to calculate individual properties in subsequent cells.")

### 1. Calculate Amino Acid Length

In [ ]:
for protein_data in protein_data_for_analysis:
    identifier = protein_data['id']
    analysed_seq = protein_data['analysis_obj']
    length = analysed_seq.length
    protein_data['properties']['length'] = length
    print(f"Protein: {identifier}, Length: {length}")

### 2. Calculate Molecular Weight

In [ ]:
for protein_data in protein_data_for_analysis:
    identifier = protein_data['id']
    analysed_seq = protein_data['analysis_obj']
    molecular_weight = analysed_seq.molecular_weight()
    protein_data['properties']['molecular_weight'] = molecular_weight
    print(f"Protein: {identifier}, Molecular Weight: {molecular_weight:.2f} Daltons")

### 3. Calculate Amino Acid Composition

In [ ]:
for protein_data in protein_data_for_analysis:
    identifier = protein_data['id']
    protein_sequence_string = protein_data['sequence'] # Get sequence string directly

    # Use collections.Counter to get amino acid counts
    amino_acid_counts = collections.Counter(protein_sequence_string)

    total_aas = sum(amino_acid_counts.values())
    if total_aas == 0:
        amino_acid_composition_percent = {aa: 0.0 for aa in amino_acid_counts.keys()}
    else:
        amino_acid_composition_percent = {aa: count / total_aas for aa, count in amino_acid_counts.items()}

    protein_data['properties']['amino_acid_composition'] = amino_acid_composition_percent
    print(f"\nProtein: {identifier}, Amino Acid Composition (percentage):")
    for aa, percentage in amino_acid_composition_percent.items():
        print(f"  {aa}: {percentage:.2%}")

### 4. Calculate Isoelectric Point (pI)

In [ ]:
for protein_data in protein_data_for_analysis:
    identifier = protein_data['id']
    analysed_seq = protein_data['analysis_obj']
    isoelectric_point = analysed_seq.isoelectric_point()
    protein_data['properties']['isoelectric_point'] = isoelectric_point
    print(f"Protein: {identifier}, Isoelectric Point (pI): {isoelectric_point:.2f}")


### Summary of All Physicochemical Properties

In [ ]:


# Extract properties into a list of dictionaries suitable for DataFrame creation
summary_data = []
for protein_data in protein_data_for_analysis:
    protein_id = protein_data['id']
    props = protein_data['properties']
    row_data = {'id': protein_id}
    row_data.update(props) # Add all calculated properties
    summary_data.append(row_data)

if summary_data:
    properties_df = pd.DataFrame(summary_data)
    # Display properties for inspection
    display(properties_df)
else:
    print("No protein properties were collected for display.")


# Calculate Sequence Similarity

In [ ]:


# Initialize the aligner
aligner = Align.PairwiseAligner()
aligner.mode = 'global' # Use global alignment (Needleman-Wunsch)
aligner.match_score = 1
aligner.mismatch_score = -1
aligner.open_gap_score = -0.5
aligner.extend_gap_score = -0.1

# Or use a substitution matrix like BLOSUM62
aligner.substitution_matrix = Align.substitution_matrices.load("BLOSUM62")

similarity_results = []
num_proteins = len(protein_data_for_analysis)

if num_proteins < 2:
    print("Need at least two protein sequences to calculate pairwise similarity.")
else:
    print(f"Calculating pairwise sequence similarity for {num_proteins} proteins...")
    for i in range(num_proteins):
        for j in range(i + 1, num_proteins):
            protein1_data = protein_data_for_analysis[i]
            protein2_data = protein_data_for_analysis[j]

            seq1_id = protein1_data['id']
            seq2_id = protein2_data['id']
            seq1 = protein1_data['sequence']
            seq2 = protein2_data['sequence']

            # Perform alignment
            alignments = aligner.align(seq1, seq2)
            # Take the best alignment score
            score = alignments.score

            similarity_results.append({
                'Protein_A_ID': seq1_id,
                'Protein_B_ID': seq2_id,
                'Similarity_Score': score
            })

    if similarity_results:
        similarity_df = pd.DataFrame(similarity_results)
        print("\nPairwise Sequence Similarity Results:")
        display(similarity_df)
    else:
        print("No similarity results to display.")


# Output for relationship between proteins
Next, we perform two types of relationship analysis:
1.  **Physicochemical Properties Clustering**: It uses the calculated length, molecular weight, and isoelectric point to cluster proteins, standardizing the data and then creating a dendrogram based on Euclidean distances.
2.  **Sequence Similarity Clustering**: It converts the previously calculated pairwise sequence similarity scores into distances and performs hierarchical clustering, displaying the relationships with another dendrogram.

In [ ]:

# Ensure protein_data_for_analysis is available and not empty
if 'protein_data_for_analysis' not in globals() or not protein_data_for_analysis:
    print("Error: No protein data found. Please run previous cells to load/analyze proteins.")
else:
    # Reconstruct properties_df from protein_data_for_analysis to ensure consistency
    summary_data = []
    for protein_data in protein_data_for_analysis:
        protein_id = protein_data['id']
        props = protein_data['properties']
        row_data = {'id': protein_id}
        row_data.update(props) # Add all calculated properties
        summary_data.append(row_data)

    if not summary_data:
        print("No protein properties were collected for analysis.")
        properties_df = pd.DataFrame() # Ensure it's defined even if empty
    else:
        properties_df = pd.DataFrame(summary_data)

    # Get ordered protein IDs for consistent indexing across analyses
    ordered_protein_ids = [p['id'] for p in protein_data_for_analysis]
    num_proteins = len(ordered_protein_ids)

    if num_proteins < 2:
        print("Need at least two protein sequences to perform clustering analyses.")
    else:
        # --- Helper function to get physicochemical distances ---
        def get_physchem_distances(properties_df, ordered_ids):
            # Ensure the properties_df is aligned with ordered_ids
            df_aligned = properties_df.set_index('id').reindex(ordered_ids).reset_index()
            # Select numerical features for clustering, excluding amino acid composition dictionary
            numeric_features_df = df_aligned[['length', 'molecular_weight', 'isoelectric_point']].copy()

            if numeric_features_df.empty or len(numeric_features_df) < 2:
                return None, None

            scaler = StandardScaler()
            scaled_features = scaler.fit_transform(numeric_features_df)
            return pdist(scaled_features, metric='euclidean'), ordered_ids

        # --- Helper function to get sequence distances ---
        def get_sequence_distances(similarity_df, ordered_ids):
            if similarity_df.empty or len(ordered_ids) < 2:
                return None, None

            num_proteins_ordered = len(ordered_ids)
            sequence_distance_matrix = np.zeros((num_proteins_ordered, num_proteins_ordered))
            id_to_index = {id_val: i for i, id_val in enumerate(ordered_ids)}

            max_score = similarity_df['Similarity_Score'].max()
            if max_score == 0:
                max_score = 1 # Avoid division by zero if all scores are 0

            for _, row in similarity_df.iterrows():
                # Only process if both proteins are in the ordered_ids list
                if row['Protein_A_ID'] in id_to_index and row['Protein_B_ID'] in id_to_index:
                    idx_a = id_to_index[row['Protein_A_ID']]
                    idx_b = id_to_index[row['Protein_B_ID']]
                    distance = max_score - row['Similarity_Score'] # Convert similarity to distance
                    sequence_distance_matrix[idx_a, idx_b] = distance
                    sequence_distance_matrix[idx_b, idx_a] = distance # Matrix is symmetric

            np.fill_diagonal(sequence_distance_matrix, 0)
            return squareform(sequence_distance_matrix), ordered_ids


        # --- 1. Relationship based on Physicochemical Properties ---
        print("\n--- Analyzing Relationships based on Physicochemical Properties ---")
        physchem_distances, physchem_labels = get_physchem_distances(properties_df, ordered_protein_ids)

        if physchem_distances is not None:
            physchem_linkage_matrix = linkage(physchem_distances, method='average')

            plt.figure(figsize=(12, 7))
            plt.title('Dendrogram of Proteins based on Physicochemical Properties')
            plt.xlabel('Protein ID')
            plt.ylabel('Distance')
            dendrogram(
                physchem_linkage_matrix,
                labels=physchem_labels,
                leaf_rotation=90.,
                leaf_font_size=8.,
            )
            plt.tight_layout()
            plt.savefig('physicochemical_dendrogram.png') # Save the plot
            plt.show()
        else:
            print("Not enough protein entries with physicochemical properties to perform clustering (at least 2 needed).")

        # --- 2. Relationship based on Sequence Similarity ---
        print("\n--- Analyzing Relationships based on Sequence Similarity ---")
        # Check if similarity_df exists in globals and is not empty
        if 'similarity_df' in globals() and not similarity_df.empty:
            sequence_distances, sequence_labels = get_sequence_distances(similarity_df, ordered_protein_ids)
            if sequence_distances is not None:
                sequence_linkage_matrix = linkage(sequence_distances, method='average')

                plt.figure(figsize=(12, 7))
                plt.title('Dendrogram of Proteins based on Sequence Similarity')
                plt.xlabel('Protein ID')
                plt.ylabel('Distance')
                dendrogram(
                    sequence_linkage_matrix,
                    labels=sequence_labels,
                    leaf_rotation=90.,
                    leaf_font_size=8.,
                )
                plt.tight_layout()
                plt.savefig('sequence_similarity_dendrogram.png') # Save the plot
                plt.show()
            else:
                print("Could not generate sequence distances. Check if similarity_df contains enough valid protein pairs or if enough proteins are loaded.")
        else:
            print("No sequence similarity data found or not enough proteins for clustering (at least 2 needed).")

        # --- 3. Relationship based on Combined Physicochemical Properties and Sequence Similarity ---
        print("\n--- Analyzing Relationships based on Combined Properties and Similarity ---")

        # Only proceed if both types of distances can be calculated for at least 2 proteins
        physchem_distances_combined, _ = get_physchem_distances(properties_df, ordered_protein_ids)
        sequence_distances_combined, _ = None, None # Initialize for the combined check

        if 'similarity_df' in globals() and not similarity_df.empty:
             sequence_distances_combined, _ = get_sequence_distances(similarity_df, ordered_protein_ids)

        if physchem_distances_combined is not None and sequence_distances_combined is not None:
            # Normalize distances to a 0-1 range before combining
            norm_physchem_dist = (physchem_distances_combined - physchem_distances_combined.min()) / (physchem_distances_combined.max() - physchem_distances_combined.min() if (physchem_distances_combined.max() - physchem_distances_combined.min()) != 0 else 1)
            norm_sequence_dist = (sequence_distances_combined - sequence_distances_combined.min()) / (sequence_distances_combined.max() - sequence_distances_combined.min() if (sequence_distances_combined.max() - sequence_distances_combined.min()) != 0 else 1)

            # Combine distances (e.g., by averaging)
            combined_distances = (norm_physchem_dist + norm_sequence_dist) / 2

            # Perform hierarchical clustering
            combined_linkage_matrix = linkage(combined_distances, method='average')

            # Plot dendrogram
            plt.figure(figsize=(12, 7))
            plt.title('Dendrogram of Proteins based on Combined Properties and Similarity')
            plt.xlabel('Protein ID')
            plt.ylabel('Distance')
            dendrogram(
                combined_linkage_matrix,
                labels=ordered_protein_ids,
                leaf_rotation=90.,
                leaf_font_size=8.,
            )
            plt.tight_layout()
            plt.savefig('combined_dendrogram.png') # Save the plot
            plt.show()
        else:
            print("Not enough data to perform combined analysis (at least 2 proteins with both physicochemical properties and sequence similarity data needed).")

# Export and Download Results

In [ ]:


print("---\nExporting and Archiving Results\n---")

# Define filenames
excel_filename = 'protein_analysis_summary.xlsx'
zip_filename = 'protein_analysis_results.zip'

# Create an Excel writer object
with pd.ExcelWriter(excel_filename, engine='xlsxwriter') as writer:
    # Export Physicochemical Properties to a sheet
    if 'properties_df' in globals() and not properties_df.empty:
        properties_df.to_excel(writer, sheet_name='Physicochemical Properties', index=False)
        print(f"'Physicochemical properties' saved to '{excel_filename}' (sheet 'Physicochemical Properties')")
    else:
        print("No 'properties_df' found or it is empty. Skipping export of physicochemical properties to Excel.")

    # Export Sequence Similarity to a sheet
    if 'similarity_df' in globals() and not similarity_df.empty:
        similarity_df.to_excel(writer, sheet_name='Sequence Similarity', index=False)
        print(f"'Sequence similarity' saved to '{excel_filename}' (sheet 'Sequence Similarity')")
    else:
        print("No 'similarity_df' found or it is empty. Skipping export of sequence similarity to Excel.")

print(f"Excel file '{excel_filename}' created.")

# Create a ZIP archive and add the Excel file and dendrogram images
with zipfile.ZipFile(zip_filename, 'w') as zf:
    zf.write(excel_filename)
    print(f"'{excel_filename}' added to '{zip_filename}'.")

    # List of dendrogram image files to include
    dendrogram_files = [
        'physicochemical_dendrogram.png',
        'sequence_similarity_dendrogram.png'
    ]

    for img_file in dendrogram_files:
        if os.path.exists(img_file):
            zf.write(img_file)
            print(f"'{img_file}' added to '{zip_filename}'.")
        else:
            print(f"Warning: Dendrogram image '{img_file}' not found. Skipping addition to ZIP.")

print(f"ZIP archive '{zip_filename}' created.")

# Download the ZIP file
files.download(zip_filename)

print("\nExport and download process complete. Please check your browser's downloads for the ZIP archive containing all results.")

# Prepare for next run

Finally, it is important to run this cell starting a new analysis without interference from previous runs. This cell provides a utility to clear all previously loaded and analyzed protein data from the current session.

In [ ]:
all_fasta_sequences = []
print("Data in 'all_fasta_sequences' has been cleared.")

protein_data_for_analysis = []
print("Data in 'protein_data_for_analysis' has been cleared.")

# Clear additional data structures for a fresh start
if 'properties_df' in globals():
    del properties_df
    print("'properties_df' has been cleared.")
if 'similarity_df' in globals():
    del similarity_df
    print("'similarity_df' has been cleared.")
